# Inspect Coverage
A quick look at the data coverage

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import seaborn as sns
import cartopy.crs as ccrs
import cartopy.feature as cfeature

DATA_DIR = Path("data/")

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted')


# CONUS extent
CONUS_EXTENT = [-125, -66, 24, 50]  # [lon_min, lon_max, lat_min, lat_max]

def make_conus_ax(fig, pos=111, title=''):
    """Return a cartopy GeoAxes set to CONUS with standard features."""
    subplot_args = pos if isinstance(pos, tuple) else (pos,)
    ax = fig.add_subplot(*subplot_args, projection=ccrs.AlbersEqualArea(
        central_longitude=-96, central_latitude=37.5))
    ax.set_extent(CONUS_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,       facecolor='#f5f5f0', zorder=0)
    ax.add_feature(cfeature.OCEAN,      facecolor='#c8e0f0', zorder=0)
    ax.add_feature(cfeature.LAKES,      facecolor='#c8e0f0', zorder=1, alpha=0.6)
    ax.add_feature(cfeature.STATES,     linewidth=0.4,       zorder=2, edgecolor='gray')
    ax.add_feature(cfeature.COASTLINE,  linewidth=0.6,       zorder=3)
    ax.add_feature(cfeature.BORDERS,    linewidth=0.6,       zorder=3)
    if title:
        ax.set_title(title)
    return ax

In [ ]:
site_info = pd.read_parquet(DATA_DIR / "metadata" / "site_info.parquet")
coverage = pd.read_parquet(DATA_DIR / "metadata" / "data_coverage.parquet")

## 1. Site Coverage

In [ ]:
tiers = [
    ("Total sites",        None),
    ("Has stage data",     "has_stage_data"),
    ("Has reach ID",       "has_reach_id"),
    ("Has action stage",   "has_action_stage"),
    ("Has minor stage",    "has_flood_stage"),
    ("Has moderate stage", "has_moderate_stage"),
    ("Has major stage",    "has_major_stage"),
]

total  = len(coverage)
counts = [total if col is None else int(coverage[col].sum()) for _, col in tiers]
labels = [t[0] for t in tiers]
pcts   = [c / total * 100 for c in counts]
rems   = [total - c for c in counts]

COLOR_HAS  = "#3a7ebf"
COLOR_MISS = "#d9e8f5"

fig, ax = plt.subplots(figsize=(8, 4), dpi=300)

bars_has  = ax.barh(labels, counts, color=COLOR_HAS,  height=0.6, label="Has attribute")
bars_miss = ax.barh(labels, rems,   left=counts, color=COLOR_MISS, height=0.6, label="Missing")

# Annotate each bar with count + percentage
for i, (c, p) in enumerate(zip(counts, pcts)):
    ax.text(c + total * 0.005, i, f"{c:,}  ({p:.0f}%)",
            va="center", ha="left", fontsize=9)

ax.set_xlim(0, total * 1.1)
ax.set_xlabel("Number of sites")
ax.set_title(f"Site coverage  |  {total:,} total sites")
ax.invert_yaxis()
# ax.legend(loc="lower right", fontsize=9)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{int(x):,}"))
sns.despine(left=True)
ax.tick_params(axis='y', length=0)

plt.grid(which='major', axis='y')
plt.tight_layout()
plt.show()

## 2. Maps of Coverage

In [ ]:
map_df = coverage.merge(
    site_info[["site_no", "latitude", "longitude"]],
    on="site_no", how="left"
).dropna(subset=["latitude", "longitude"])

### 2.1 All Sites

In [ ]:
fig = plt.figure(figsize=(6, 4), dpi=300)

ax = make_conus_ax(fig, pos=(1,1,1))
sc = ax.scatter(
    map_df['longitude'], map_df['latitude'],
    c='cornflowerblue',
    s=5, alpha=0.8, linewidths=0,
    transform=ccrs.PlateCarree(), zorder=4
)

fig.suptitle(f'All Sites  |  n={len(map_df)}', fontsize=10)
plt.tight_layout()
plt.show()

### 2.2 Has Stage

In [ ]:
fig = plt.figure(figsize=(6, 4), dpi=300)

ax = make_conus_ax(fig, pos=(1,1,1))
ax.scatter(
    map_df['longitude'][map_df['has_stage_data']==False], map_df['latitude'][map_df['has_stage_data']==False],
    c='cornflowerblue',
    s=5, alpha=0.5, linewidths=0,
    label=f'Discharge only  |  n={len(map_df) - sum(map_df["has_stage_data"])}',
    transform=ccrs.PlateCarree(), zorder=4
)
ax.scatter(
    map_df['longitude'][map_df['has_stage_data']==True], map_df['latitude'][map_df['has_stage_data']==True],
    c='darkorange',
    s=5, alpha=0.8, linewidths=0,
    label=f'Discharge and stage  |  n={sum(map_df["has_stage_data"])}',
    transform=ccrs.PlateCarree(), zorder=4
)

ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.04),
    ncol=2,
    markerscale=2,
    fontsize=8,
    frameon=False,
)

plt.tight_layout()
plt.show()

### 2.3 Has all flood thresholds

In [ ]:
# all_flood = map_df[
#     (map_df['has_flood_stage'] == True) &
#     (map_df['has_action_stage'] == True) &
#     (map_df['has_moderate_stage'] == True) &
#     (map_df['has_major_stage'] == True)]

map_df['all_thresh'] = sum((
    map_df['has_flood_stage'],
    map_df['has_action_stage'],
    map_df['has_moderate_stage'],
    map_df['has_major_stage']
))

In [ ]:
fig = plt.figure(figsize=(6, 4), dpi=300)

ax = make_conus_ax(fig, pos=(1,1,1))
ax.scatter(
    map_df['longitude'][map_df['all_thresh']==0], map_df['latitude'][map_df['all_thresh']==0],
    c='cornflowerblue',
    s=5, alpha=0.5, linewidths=0,
    label=f'Missing at least 1 flood stage  |  n={len(map_df['longitude'][map_df['all_thresh']==0])}',
    transform=ccrs.PlateCarree(), zorder=4
)
ax.scatter(
    map_df['longitude'][map_df['all_thresh']==4], map_df['latitude'][map_df['all_thresh']==4],
    c='darkorange',
    s=5, alpha=0.8, linewidths=0,
    label=f'Has all flood stages  |  n={len(map_df['longitude'][map_df['all_thresh']==4])}',
    transform=ccrs.PlateCarree(), zorder=4
)

ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, -0.04),
    ncol=2,
    markerscale=2,
    fontsize=8,
    frameon=False,
)

plt.tight_layout()
plt.show()

## 2.b. Only basins with DA in range

In [ ]:
a = map_df[map_df['all_thresh'] == 4]